# RoadAlert — Genuine RDD2022 India YOLO Training Pipeline (Google Colab GPU)

This notebook implements an end-to-end, reproducible training and evaluation pipeline for **Road Damage Detection (RDD)** on Indian roads using the official **RDD2022 India** dataset (IEEE BigData Cup / CRDDC 2022).

### Key Objectives:
1. **Verified Dataset Acquisition:** Download and verify the official RDD2022 India dataset archive.
2. **Data Integrity Audit:** Parse Pascal VOC XML annotations, confirm genuine classes (`D00`, `D10`, `D20`, `D40`), and verify image-label pairing.
3. **Leakage-Aware Splitting:** Group contiguous frame sequences into Train (80%), Validation (10%), and Test (10%) splits to prevent sequential dashcam frame leakage.
4. **YOLOv8 Fine-Tuning:** Train Ultralytics YOLOv8s with full GPU acceleration.
5. **Real Evaluation:** Evaluate on the held-out test split and report genuine mAP, Precision, Recall, and Confusion Matrix.
6. **Export for RoadAlert:** Export the trained weights to ONNX (`best.onnx`) for integration with RoadAlert's Node.js backend.

> **Important Note:** Physical road damage detection identifies visible potholes and cracks. It provides visual evidence for incident corroboration in RoadAlert, but does not independently establish incident authenticity or timestamp truth.

## 1. Colab GPU Verification
Ensure that this notebook is running on a GPU runtime (T4, V100, or A100).

In [ ]:
# Verify GPU acceleration in Google Colab
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {device_name} ({total_memory:.2f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Please navigate to 'Runtime -> Change runtime type' and select 'T4 GPU'.")

!nvidia-smi

## 2. Install Required Dependencies
Install `ultralytics` for YOLOv8/YOLO11, `pyyaml` for dataset configuration, and XML parsing utilities.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q "ultralytics>=8.1.0" "pyyaml>=6.0" "tqdm>=4.66.0" "matplotlib>=3.8.0"

import ultralytics
ultralytics.checks()

## 3. Official RDD2022 India Dataset Acquisition & Extraction

The **RDD2022** dataset was released through the CRDDC'2022 (Crowdsensing-based Road Damage Detection Challenge) hosted on **Figshare** (DOI: `10.6084/m9.figshare.21431547`).

### Option A: Direct Download in Colab
We download the official dataset archive and extract the `India` country folder (`RDD2022/India/train`).

### Option B: Google Drive Mount (Recommended for saving quota)
If you already saved the archive to Google Drive, you can mount Drive directly.

In [ ]:
import os
import sys
import shutil
import zipfile
import tarfile
from pathlib import Path

WORKSPACE_DIR = Path("/content/rdd2022_workspace")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/rdd2022_workspace

INDIA_RAW_DIR = WORKSPACE_DIR / "India"

# Check if dataset is already extracted
if (INDIA_RAW_DIR / "train" / "images").exists() and (INDIA_RAW_DIR / "train" / "annotations" / "xmls").exists():
    print("RDD2022 India dataset already present in workspace.")
else:
    print("Downloading official RDD2022 dataset metadata and files from Figshare...")
    # Download label map and directory structure for verification
    !wget -q -nc https://ndownloader.figshare.com/files/38030820 -O label_map.pbtxt
    !wget -q -nc https://ndownloader.figshare.com/files/38030823 -O Directory_Structure.txt
    
    # Figshare Full Archive URL: https://ndownloader.figshare.com/files/38030910 (~12.6 GB total package)
    # Alternatively, direct download using gdown or direct mirror:
    print("\n--- Dataset Acquisition Options ---")
    print("1. Full Figshare Bundle: https://ndownloader.figshare.com/files/38030910")
    print("2. Google Drive: If you have 'RDD2022_India.zip' in Drive, uncomment the Drive mount below.")
    
    # To mount Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # shutil.copy('/content/drive/MyDrive/RDD2022_India.zip', '/content/rdd2022_workspace/')
    
    # Direct automated download of CRDDC 2022 India package:
    !wget -c https://ndownloader.figshare.com/files/38030910 -O RDD2022.zip
    
    # Extract ONLY the India folder to save disk space and time
    print("Extracting India subset from RDD2022.zip...")
    !unzip -q -o RDD2022.zip "RDD2022/India/*" -d .
    if (WORKSPACE_DIR / "RDD2022" / "India").exists():
        shutil.move(str(WORKSPACE_DIR / "RDD2022" / "India"), str(INDIA_RAW_DIR))
        shutil.rmtree(str(WORKSPACE_DIR / "RDD2022"), ignore_errors=True)
    # Clean up zip to free 12GB disk space
    if os.path.exists("RDD2022.zip"):
        os.remove("RDD2022.zip")

print("Extraction check finished.")

## 4. Verification & Exploratory Data Analysis (EDA)
**Crucial Step:** We must verify that images and XML annotation files exist on disk before writing any training configs. We parse the XML files to discover the exact classes actually present in the data.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter
import glob

train_images_dir = INDIA_RAW_DIR / "train" / "images"
train_xmls_dir = INDIA_RAW_DIR / "train" / "annotations" / "xmls"

assert train_images_dir.exists(), f"Images directory missing: {train_images_dir}"
assert train_xmls_dir.exists(), f"Annotations directory missing: {train_xmls_dir}"

image_files = sorted(list(train_images_dir.glob("*.jpg")) + list(train_images_dir.glob("*.png")))
xml_files = sorted(list(train_xmls_dir.glob("*.xml")))

print(f"Detected Training Images: {len(image_files)}")
print(f"Detected Annotation XMLs: {len(xml_files)}")

assert len(image_files) > 0, "Zero images found! Verify dataset extraction."
assert len(xml_files) > 0, "Zero XML annotations found! Verify dataset extraction."

# Scan XML annotations for class distribution
class_counts = Counter()
box_dimensions = []
corrupted_xmls = []

for xml_path in xml_files:
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall("object"):
            cls_name = obj.find("name").text.strip()
            class_counts[cls_name] += 1
    except Exception as e:
        corrupted_xmls.append((xml_path.name, str(e)))

print(f"\nCorrupted XML files: {len(corrupted_xmls)}")
print("\nActual Class Distribution in RDD2022 India Annotations:")
for cls_name, count in class_counts.most_common():
    print(f"  - {cls_name}: {count} bounding boxes")

# Define official RDD2022 class mapping
RDD_CLASSES = ["D00", "D10", "D20", "D40"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(RDD_CLASSES)}
CLASS_DESCRIPTIONS = {
    "D00": "Longitudinal Crack",
    "D10": "Transverse / Lateral Crack",
    "D20": "Alligator (Fatigue) Mesh Crack",
    "D40": "Pothole / Rutting / Bump"
}

print("\nTarget YOLO Classes:")
for idx, name in enumerate(RDD_CLASSES):
    print(f"  ID {idx}: {name} ({CLASS_DESCRIPTIONS.get(name)})")

## 5. Pascal VOC XML to YOLO Format Conversion
YOLO requires normalized bounding boxes in text files formatted as:
`<class_id> <x_center> <y_center> <width> <height>` (all values normalized between 0.0 and 1.0).

In [ ]:
YOLO_LABELS_DIR = WORKSPACE_DIR / "yolo_labels"
YOLO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

converted_count = 0
empty_count = 0

for xml_path in xml_files:
    txt_name = xml_path.stem + ".txt"
    txt_path = YOLO_LABELS_DIR / txt_name
    
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    size = root.find("size")
    width = float(size.find("width").text)
    height = float(size.find("height").text)
    
    lines = []
    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in CLASS_TO_IDX:
            continue  # Ignore non-standard classes if any
            
        cls_id = CLASS_TO_IDX[cls_name]
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        
        # Clip to image boundaries
        xmin = max(0.0, min(width, xmin))
        xmax = max(0.0, min(width, xmax))
        ymin = max(0.0, min(height, ymin))
        ymax = max(0.0, min(height, ymax))
        
        bw = xmax - xmin
        bh = ymax - ymin
        if bw <= 0 or bh <= 0:
            continue
            
        x_center = (xmin + bw / 2.0) / width
        y_center = (ymin + bh / 2.0) / height
        norm_w = bw / width
        norm_h = bh / height
        
        lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
        
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
        
    if lines:
        converted_count += 1
    else:
        empty_count += 1

print(f"Conversion complete: {converted_count} files with annotations, {empty_count} background/empty files.")

## 6. Leakage-Aware Dataset Splitting

### Understanding Leakage in Road Damage Imagery:
- **Metadata Availability Assessment:** RDD2022 India does **not** provide separate video trip IDs or GPS track files. The available metadata consists of image filenames (e.g. `India_000004.jpg`, `India_000006.jpg`) and VOC XML files.
- **Temporal Frame Correlation:** In dashcam road recordings, consecutive numbered frames (`India_000004`, `India_000006`, `India_000008`) capture nearly identical scenes from the moving vehicle.
- **Leakage-Aware Strategy:** Rather than random uniform shuffling (which spreads adjacent video frames across train and validation sets), we sort images by frame sequence and assign **contiguous blocks of 100 frames** into Train (80%), Validation (10%), and Test (10%). This ensures entire driving segments remain within a single split.

In [ ]:
import random
import yaml

DATASET_ROOT = WORKSPACE_DIR / "yolo_dataset"
for split in ["train", "val", "test"]:
    (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Pair images with existing label files
paired_files = []
for img_path in image_files:
    label_path = YOLO_LABELS_DIR / (img_path.stem + ".txt")
    if label_path.exists():
        paired_files.append((img_path, label_path))

print(f"Total Valid Image-Label Pairs: {len(paired_files)}")

# Sort files by sequence index to group consecutive video frames
paired_files.sort(key=lambda x: x[0].stem)

# Contiguous Chunk Splitting (Chunk size = 100 frames to avoid splitting video passes)
CHUNK_SIZE = 100
chunks = [paired_files[i:i + CHUNK_SIZE] for i in range(0, len(paired_files), CHUNK_SIZE)]

# Deterministic block assignment: 80% train, 10% val, 10% test
train_pairs = []
val_pairs = []
test_pairs = []

for i, chunk in enumerate(chunks):
    mod = i % 10
    if mod == 8:
        val_pairs.extend(chunk)
    elif mod == 9:
        test_pairs.extend(chunk)
    else:
        train_pairs.extend(chunk)

print(f"Split Counts:")
print(f"  - Train: {len(train_pairs)} pairs ({len(train_pairs)/len(paired_files)*100:.1f}%)")
print(f"  - Val:   {len(val_pairs)} pairs ({len(val_pairs)/len(paired_files)*100:.1f}%)")
print(f"  - Test:  {len(test_pairs)} pairs ({len(test_pairs)/len(paired_files)*100:.1f}%)")

# Create symlinks / copy files into YOLO directory structure
def populate_split(pairs, split_name):
    for img, lbl in pairs:
        target_img = DATASET_ROOT / "images" / split_name / img.name
        target_lbl = DATASET_ROOT / "labels" / split_name / lbl.name
        if not target_img.exists():
            os.symlink(img, target_img)
        if not target_lbl.exists():
            os.symlink(lbl, target_lbl)

populate_split(train_pairs, "train")
populate_split(val_pairs, "val")
populate_split(test_pairs, "test")

# Generate data.yaml
data_yaml_content = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(RDD_CLASSES),
    "names": RDD_CLASSES
}

DATA_YAML_PATH = WORKSPACE_DIR / "rdd_india.yaml"
with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print(f"Generated data.yaml at: {DATA_YAML_PATH}")

## 7. Ultralytics YOLOv8 Training
We initialize with pretrained `yolov8s.pt` weights and fine-tune on the verified RDD2022 India data.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-Small backbone
model = YOLO("yolov8s.pt")

# Train model with real hyperparameters
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    device=0,
    project="RoadAlert_RDD_Runs",
    name="yolov8s_rdd2022_india",
    save=True,
    plots=True
)

print("Training finished successfully!")

## 8. Real Model Evaluation on Held-Out Test Split
**Strict Rule:** No fabricated metrics. We evaluate the best checkpoint (`best.pt`) on the held-out test split and extract the exact computed validation metrics.

In [ ]:
# Load best trained checkpoint
best_weights = Path("RoadAlert_RDD_Runs/yolov8s_rdd2022_india/weights/best.pt")
assert best_weights.exists(), f"Trained weights not found: {best_weights}"

trained_model = YOLO(str(best_weights))

# Evaluate strictly on held-out test split
metrics = trained_model.val(data=str(DATA_YAML_PATH), split="test")

print("\n--- Actual Evaluation Metrics on RDD2022 India Test Split ---")
print(f"  Mean Precision:  {metrics.box.mp:.4f}")
print(f"  Mean Recall:     {metrics.box.mr:.4f}")
print(f"  mAP@0.5:         {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:    {metrics.box.map:.4f}")

print("\nPer-Class AP@0.5:")
for idx, cls_name in enumerate(RDD_CLASSES):
    ap = metrics.box.ap50[idx] if idx < len(metrics.box.ap50) else 0.0
    print(f"  - {cls_name} ({CLASS_DESCRIPTIONS.get(cls_name)}): {ap:.4f}")

## 9. Export to ONNX for RoadAlert Backend
Export the model to Open Neural Network Exchange (ONNX) format with dynamic input shape for high-performance inference in Node.js / Express.

In [ ]:
# Export to ONNX
onnx_path = trained_model.export(format="onnx", dynamic=True, simplify=True)
print(f"Exported ONNX Model: {onnx_path}")

# Display output files for download
!ls -lh RoadAlert_RDD_Runs/yolov8s_rdd2022_india/weights/

## 10. Sample Test Visualizer
Run test inference on sample images and display bounding boxes with detected road damage categories.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Pick 3 test images
test_sample_imgs = list((DATASET_ROOT / "images" / "test").glob("*.jpg"))[:3]
results = trained_model.predict(source=test_sample_imgs, conf=0.25, save=False)

for i, r in enumerate(results):
    plotted = r.plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(plotted)
    plt.axis("off")
    plt.title(f"Test Sample {i+1}: {len(r.boxes)} damage defects detected")
    plt.show()

## 11. RoadAlert Backend Integration Guide

### How to use `best.onnx` in RoadAlert Backend:
1. Download `best.onnx` from the Colab weights directory: `RoadAlert_RDD_Runs/yolov8s_rdd2022_india/weights/best.onnx`.
2. Place `best.onnx` into `backend/models/road_damage_yolov8s.onnx`.
3. In RoadAlert's `backend/src/services/incidentService.ts` (`attachIncidentEvidence`):
```typescript
// Example integration hook (to be enabled when weights are ready):
// When a user uploads a PHOTO to /api/v1/incidents/:id/evidence:
// 1. Pass image buffer to onnxruntime-node
// 2. If D40 (pothole) or D20 (alligator crack) is detected with conf >= 0.40:
//    Attach damage metadata: { damage_type: 'D40', confidence: 0.82 }
// 3. VerificationEngine awards +15 validated multimedia evidence points.
```